In [1]:
import numpy as np
import pickle
from scipy import io as sio
import c_solver_2D as csolver

PHCv2.4.86 released 2022-05-30 works!


In [62]:
from scipy import spatial
from scipy import linalg as la

In [2]:
import matplotlib.pyplot as plt

In [4]:
%matplotlib

Using matplotlib backend: QtAgg


In [39]:
import os

In [79]:
import BerryPy.TightBinding as tb

# 2D topology

In [31]:
with open("data/GBZ_data_6/x_major_1.pkl", "rb") as fp:
    all_segs = pickle.load(fp)

In [40]:
os.mkdir("data/GBZ_data_6_filtered")

In [43]:
for j in range(50):
    with open("data/GBZ_data_6/x_major_%d.pkl"%(j), "rb") as fp:
        all_segs = pickle.load(fp)
    all_long_segs = []
    for seg in all_segs:
        if(seg.shape[0] > 1):
            all_long_segs.append(seg)
    with open("data/GBZ_data_6/x_major_%d_filtered.pkl"%(j), "wb") as fp:
        pickle.dump(all_long_segs, fp)
    with open("data/GBZ_data_6/y_major_%d.pkl"%(j), "rb") as fp:
        all_segs = pickle.load(fp)
    all_long_segs = []
    for seg in all_segs:
        if(seg.shape[0] > 1):
            all_long_segs.append(seg)
    with open("data/GBZ_data_6/y_major_%d_filtered.pkl"%(j), "wb") as fp:
        pickle.dump(all_long_segs, fp)
    

In [48]:
fig = plt.figure()
ax = fig.gca()
N_total_segs = 0
upper_band = []
lower_band = []
for j in range(50):
    with open("data/GBZ_data_6/x_major_%d_filtered.pkl"%(j), "rb") as fp:
        all_segs = pickle.load(fp)
    for seg in all_segs:
        N_total_segs += 1
        ax.plot(seg[:,0].real, seg[:,0].imag)
        if(seg[0,0].real > 0):
            upper_band.append(seg)
        else:
            lower_band.append(seg)
plt.show()

In [49]:
print(N_total_segs)
print(len(upper_band))
print(len(lower_band))

13404
6676
6728


In [51]:
for seg in upper_band:
    plt.plot(seg[:,0].real, seg[:,0].imag, 'r.')
for seg in lower_band:
    plt.plot(seg[:,0].real, seg[:,0].imag, 'b.')
plt.show()

In [52]:
upper_band_arr = np.vstack(upper_band)
lower_band_arr = np.vstack(lower_band)

In [264]:
OBC_data = sio.loadmat("data/OBC_spectrum_6_N150_N250.mat")
eigv = OBC_data['eigv'].flatten()
plt.plot(eigv.real, eigv.imag, '.')

In [266]:
plt.plot(upper_band_arr[:,0].real, upper_band_arr[:,0].imag, 'b.')
plt.plot(lower_band_arr[:,0].real, lower_band_arr[:,0].imag, 'r.')
plt.show()

In [54]:
upper_band_k1 = np.log(upper_band_arr[:,1]).imag
upper_band_k2 = np.log(upper_band_arr[:,2]).imag
lower_band_k1 = np.log(lower_band_arr[:,1]).imag
lower_band_k2 = np.log(lower_band_arr[:,2]).imag


In [71]:
A = np.array([
    [1,2],
    [3,4],
    [5,6],
    [7,8]
])
print(A + np.array([1,1]))

[[2 3]
 [4 5]
 [6 7]
 [8 9]]


In [241]:
def toric_Delaunay(points, X_period, Y_period):
    N_points = points.shape[0]
    # shifts
    enlarged_points = np.vstack([
        points,
        points + np.array([X_period,0]),
        points + np.array([0, Y_period]),
        points + np.array([X_period,Y_period]),
        points + np.array([-X_period,0]),
        points + np.array([0, -Y_period]),
        points + np.array([-X_period,Y_period]),
        points + np.array([X_period, -Y_period]),
        points + np.array([-X_period, -Y_period])
    ])
    enlarged_tri = spatial.Delaunay(enlarged_points)

    # plt.triplot(enlarged_points[:,0], enlarged_points[:,1], enlarged_tri.simplices)
    # plt.plot([-X_period/2, -X_period/2, X_period/2, X_period/2, -X_period/2], 
    #          [-Y_period/2, Y_period/2, Y_period/2, -Y_period/2, -Y_period/2], 'k-')

    toric_simplices = []
    # toric_simplices_extended = []
    for curr_simplex in enlarged_tri.simplices:
        curr_regions = set([j//N_points for j in curr_simplex])
        cases = [
            (0 in curr_regions) and (0 in curr_regions),
            (0 in curr_regions) and (1 in curr_regions),
            (0 in curr_regions) and (2 in curr_regions),
            (0 in curr_regions) and (3 in curr_regions),
            (1 in curr_regions) and (2 in curr_regions)
        ]
        if(any(cases) and curr_regions.issubset({0,1,2,3})):
            curr_normalized_simplex = [(curr_index % N_points) for curr_index in curr_simplex]
            toric_simplices.append(curr_normalized_simplex)
            # toric_simplices_extended.append(list(curr_simplex))

    #     if(any(np.asarray(curr_simplex) < N_points)):
    #         if(not all(np.asarray(curr_simplex) < N_points)):
    #             curr_normalized_simplex = [(curr_index % N_points) for curr_index in curr_simplex]
    #             if(simplices_lookup.count(set(curr_normalized_simplex)) == 0):
    #                 toric_simplices.append(curr_normalized_simplex)
    #                 simplices_lookup.append(set(curr_normalized_simplex))
    #                 toric_simplices_extended.append(list(curr_simplex))
    #         else:
    #             toric_simplices.append(list(curr_simplex))
    #             simplices_lookup.append(set(curr_simplex))
    #             toric_simplices_extended.append(list(curr_simplex))
    # plt.triplot(enlarged_points[:,0], enlarged_points[:,1], toric_simplices_extended)
    # plt.show()
    return toric_simplices

In [55]:
plt.plot(upper_band_k1, upper_band_k2, '.')

In [135]:
upper_band_tri = spatial.Delaunay(np.column_stack([upper_band_k1, upper_band_k2]))

In [240]:
upper_band_toric = toric_Delaunay(np.column_stack([upper_band_k1, upper_band_k2]), 2 * np.pi, 2 * np.pi)

In [244]:
lower_band_toric = toric_Delaunay(np.column_stack([lower_band_k1, lower_band_k2]), 2 * np.pi, 2 * np.pi)

In [64]:
for curr_simplex in upper_band_tri.simplices:
    curr_det = np.array(
        [
            [upper_band_k1[curr_simplex[1]] - upper_band_k1[curr_simplex[0]], upper_band_k1[curr_simplex[2]] - upper_band_k1[curr_simplex[0]]],
            [upper_band_k2[curr_simplex[1]] - upper_band_k2[curr_simplex[0]], upper_band_k2[curr_simplex[2]] - upper_band_k2[curr_simplex[0]]]
        ]
    )
    if(la.det(curr_det) < 0):
        print(la.det(curr_det))

In [75]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
# ax = fig.gca()
# ax.plot_trisurf(upper_band_k1, upper_band_k2, upper_band_arr[:,0].real, triangles = upper_band_tri.simplices)
ax.plot_trisurf(upper_band_k1, upper_band_k2, upper_band_arr[:,0].real, triangles = upper_band_toric)
plt.show()

In [268]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
b = 5
a = 1
X = (b + a * np.cos(upper_band_k2)) * np.cos(upper_band_k1)
Y = (b + a * np.cos(upper_band_k2)) * np.sin(upper_band_k1)
Z = a * np.sin(upper_band_k2)
# ax.plot(upper_band_k1, upper_band_k2, upper_band_arr[:,0].real, '.')
# ax.plot_trisurf(X, Y, Z, triangles = upper_band_tri.simplices)
ax.plot_trisurf(X, Y, Z, triangles = upper_band_toric, color=(0,0,0,0), edgecolor=(0,0,0,1))
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

[]

In [65]:
with open("data/GBZ_data_7/x_major_30.pkl", "rb") as fp:
    all_segs = pickle.load(fp)

In [66]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for seg in all_segs:
    ax.plot(seg[:,1].real, seg[:,1].imag, seg[:,0].real)
plt.show()

In [32]:
print(len(all_segs))

1204


In [33]:
one_point_segs = []
for seg in all_segs:
    if(seg.shape[0] == 1):
        one_point_segs.append(seg)

In [34]:
print(len(one_point_segs))

736


In [38]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
print(len(all_segs))
for seg in all_segs:
    ax.plot(np.log(seg[:,1]).imag,np.log(seg[:,2]).imag , seg[:,0].real)
for seg in one_point_segs:
    ax.plot(np.log(seg[:,1]).imag,np.log(seg[:,2]).imag , seg[:,0].real, '.')
    # ax.plot(seg[:,1].real, seg[:,1].imag, seg[:,0].real)
plt.show()

1204


In [27]:
loops, fragments = csolver.segments_to_loops(all_segs, 0.5, 10)

In [28]:
print(len(loops))
print(len(fragments))

18
240


In [30]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for curr_loop in loops:
    # ax.plot(curr_loop[:,1].real, curr_loop[:,1].imag, curr_loop[:,0].real)
    ax.plot(np.log(curr_loop[:,1]).imag,np.log(curr_loop[:,2]).imag , curr_loop[:,0].real)
    print(curr_loop.shape[0])
    # plt.pause(1)
plt.show()


199
400
400
203
150
307
53
174
42
24
70
18
120
36
18
136
26
24


In [11]:
for curr_loop in loops:
    print(csolver.winding_number(curr_loop[:,1]))

2
2
2
2


In [109]:
def get_eigenstates(model:tb.TightBindingModel, GBZ_point):
    H = model.get_bulk_Hamiltonian_dense(np.log(GBZ_point[1:])/(2j*np.pi))
    eigv_eq = np.eye(H.shape[0]) * GBZ_point[0] - H
    eigvec = la.null_space(eigv_eq, rcond=1e-6)
    # print(la.det(eigv_eq))
    # print(eigvec)
    return eigvec

In [110]:
def winding_phase_fun(model:tb.TightBindingModel, vec_list:list):
    winding_phase_multiplier = 1
    # vec_list = [get_eigenstates(model, point_list[j,:])[:,0] for j in loop_indices]
    for j in range(len(vec_list) - 1):
        winding_phase_multiplier *= (vec_list[j].conjugate()).dot(vec_list[j+1])
    winding_phase_multiplier *= (vec_list[-1].conjugate()).dot(vec_list[0])
    return -np.log(winding_phase_multiplier).imag

In [111]:
def Chern_number(model:tb.TightBindingModel, point_list:np.ndarray, simplices:list):
    Berry_flux = 0
    all_vecs = [get_eigenstates(model, point_list[j,:])[:,0] for j in range(point_list.shape[0])]
    for curr_simplex in simplices:
        curr_vec_list = [all_vecs[j] for j in curr_simplex]
        Berry_flux += winding_phase_fun(model, curr_vec_list)
    return Berry_flux / (2*np.pi)

In [195]:
print(Chern_number(model, upper_band_arr, upper_band_tri.simplices))

-0.08631383151736598


In [243]:
print(Chern_number(model, upper_band_arr, upper_band_toric))

-0.9999999999999962


In [245]:
print(Chern_number(model, lower_band_arr, lower_band_toric))

0.9999999999999972


In [126]:
def area_calculation(k1_list, k2_list, toric_simplices:list):
    all_area = 0
    for curr_simplex in toric_simplices:
        p1x = k1_list[curr_simplex[1]] - k1_list[curr_simplex[0]]
        p_list = [p1x - 2 * np.pi, p1x, p1x + 2 * np.pi]
        min_p = np.argmin(np.abs(p_list))
        p1x = p_list[min_p]

        p1y = k2_list[curr_simplex[1]] - k2_list[curr_simplex[0]]
        p_list = [p1y - 2 * np.pi, p1y, p1y + 2 * np.pi]
        min_p = np.argmin(np.abs(p_list))
        p1y = p_list[min_p]

        p2x = k1_list[curr_simplex[2]] - k1_list[curr_simplex[0]]
        p_list = [p2x - 2 * np.pi, p2x, p2x + 2 * np.pi]
        min_p = np.argmin(np.abs(p_list))
        p2x = p_list[min_p]

        p2y = k2_list[curr_simplex[2]] - k2_list[curr_simplex[0]]
        p_list = [p2y - 2 * np.pi, p2y, p2y + 2 * np.pi]
        min_p = np.argmin(np.abs(p_list))
        p2y = p_list[min_p]

        all_area += la.det([
            [p1x, p2x],
            [p1y, p2y]
        ])/2
    return all_area

In [242]:
print(area_calculation(upper_band_k1, upper_band_k2, upper_band_toric))

39.4784176043435


In [119]:
# plt.plot(upper_band_k1/np.pi, upper_band_k2/np.pi, '.')
plt.triplot(np.column_stack([upper_band_k1/np.pi, upper_band_k2/np.pi]))

In [122]:
print((2*np.pi) **2)

39.47841760435743


In [81]:
with open("data/GBZ_model_6.pkl", "rb") as fp:

    model = pickle.load(fp)

In [88]:
eigvec = get_eigenstates(model, upper_band_arr[0,:])

(4.440172665480811e-17+6.5527436948497284e-18j)


In [90]:
print(la.norm(eigvec))

0.9999999999999998


In [269]:
N_points = 300
k1_list = np.random.rand(N_points) * (2 * np.pi) - np.pi
k2_list = np.random.rand(N_points) * (2 * np.pi) - np.pi
plt.plot(k1_list, k2_list, '.')
plt.xlabel("$\\mathrm{Im}\\log(\\beta_1)$")
plt.ylabel("$\\mathrm{Im}\\log(\\beta_2)$")

Text(0, 0.5, '$\\mathrm{Im}\\log(\\beta_2)$')

In [270]:
toric_test = toric_Delaunay(np.column_stack([k1_list, k2_list]), 2 * np.pi, 2 * np.pi)

In [275]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
b = 5
a = 1
X = (b + a * np.cos(k2_list)) * np.cos(k1_list)
Y = (b + a * np.cos(k2_list)) * np.sin(k1_list)
Z = a * np.sin(k2_list)
# ax.plot(k1_list, k2_list, upper_band_arr[:,0].real, '.')
# ax.plot_trisurf(X, Y, Z, triangles = upper_band_tri.simplices)
ax.plot_trisurf(X, Y, Z, triangles = toric_test, color=(0,0,0,0), edgecolor=(0,0,0.5,1))
ax.set_aspect('equal')
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

[]

In [184]:
plt.triplot(k1_list, k2_list, toric_test)

In [188]:
plane_test = spatial.Delaunay(np.column_stack([k1_list, k2_list]))

In [239]:
print(area_calculation(k1_list, k2_list, toric_test))
# print(area_calculation(k1_list, k2_list, plane_test.simplices))

39.47841760435746


In [216]:
(2*np.pi) ** 2

39.47841760435743

# Try CGAL

In [197]:
import cgal.periodic_2d_triangulation as cgal_pt

ModuleNotFoundError: No module named 'cgal'

# Minimal winding

In [248]:
with open("data/GBZ_data_8/y_major_0.pkl", "rb") as fp:
    all_segments = pickle.load(fp)

In [251]:
data = sio.loadmat("data/GBZ_char_poly_8.mat")
coeffs = data['coeffs'].flatten()
degrees = data['orders'].flatten()

In [253]:
char_poly = csolver.CLaurant(3)
char_poly.set_Laurant_by_terms(csolver.CScalarVec(coeffs), csolver.CIntVec(degrees))

In [254]:
print(char_poly.num.to_str(csolver.CStrVec(["E", "x", "y"])))

+(-1+0*i)*(y**2)+(-1+0*i)*x*y+(-1+0*i)*(x**2)+(-1+0*i)*(x**2)*(y**2)+(-1+0*i)*(x**3)+(-1+0*i)*(x**3)*y+(1+0*i)*E*(x**2)*y;


In [255]:
print(char_poly.denom_orders)

aGBZ_2D_solver_IndexVec[0, 2, 1]


In [256]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for seg in all_segments:
    ax.plot(seg[:,2].real, seg[:,2].imag, seg[:,0].real)
plt.show()